# Dates and seasons

This lesson collects the techniques that you need for the course project, and practises them on a small dataset:

- converting date and time stamps into real dates with `pd.to_datetime()`,
- assigning each month to a **season**, including the tricky December,
- grouping by **two keys** (season-year and season) and leaving out **incomplete seasons**,
- and drawing one panel per season with a `for` loop.

### Input data

We use the simulated weather observations for five Swedish cities from Lecture 6.1 (the `dataset` folder): four observations per day from 2018 to 2020, temperatures in degrees Fahrenheit. The file `dataset/README.md` describes the data.

### Sources

This lesson was written for this course. It builds on Lectures 3.4, 6.1, 7.1 and 7.2.

In [ ]:
# pandas for the data, matplotlib for the plots
import pandas as pd
import matplotlib.pyplot as plt

## Parsing dates

Let's read the Stockholm file. As in Lecture 6.1, the columns are separated by whitespace and missing values are written as `*` characters.

In [ ]:
# The * characters that mean "no data" in these files
na_values = ["*", "**", "***", "****"]

data = pd.read_csv(
    "dataset/Stockholm_continuous_final.txt",
    sep=r"\s+",
    usecols=["YRMODAHRMN", "Temperature"],
    na_values=na_values,
)
data.head()

In Lecture 6.1 we cut the year and the month out of the `YRMODAHRMN` time stamps with string slicing. pandas can also turn the time stamps into real **dates** with `pd.to_datetime()`. The `format` text tells pandas where each part of the date is found:

| Code | Meaning | Example |
| --- | --- | --- |
| `%Y` | year, four digits | 2018 |
| `%m` | month, two digits | 07 |
| `%d` | day of the month | 15 |
| `%H` | hour (24-hour clock) | 18 |
| `%M` | minute | 00 |

Our time stamps look like `201807151800`, so the format is `"%Y%m%d%H%M"`. The format describes the time stamps as text, so we first convert the numbers to text with `.astype(str)`. (pandas would also accept the numbers directly, but this makes the step easier to follow.)

In [ ]:
# Convert the time stamps (numbers) to text, and then to dates
data["TIME"] = pd.to_datetime(data["YRMODAHRMN"].astype(str), format="%Y%m%d%H%M")

data.head()

In [ ]:
# The new column has a date-and-time data type
data.dtypes

> ⚠️ **Common Pitfall: `%m` is the month, `%M` is the minute**
>
> The two codes differ only in the capital letter. With `format="%Y%M%d%H%m"` pandas would read the month as minutes and the minutes as the month. For our data this stops with an error, but with other data it can silently produce wrong dates. Always check a few converted values against the original text.

Once a column holds dates, the `.dt` *accessor* gives us the parts of each date as numbers: `.dt.year`, `.dt.month`, `.dt.day`, `.dt.hour` and more. Let's add the year and the month as new columns, and convert the temperature to Celsius:

In [ ]:
# Year and month of each observation
data["YEAR"] = data["TIME"].dt.year
data["MONTH"] = data["TIME"].dt.month

# Temperature in degrees Celsius
data["TEMP_C"] = (data["Temperature"] - 32) / 1.8

data.head()

> **Note:** the project data stores dates as whole numbers in the form `YYYYMMDD` (for example `18490101`). The same method works there, with `format="%Y%m%d"`.

## From observations to monthly means

The project data contains one value per month. Our data has four values per day, so we first calculate the mean temperature of every month by grouping on **two** columns, `YEAR` and `MONTH` (Lecture 6.1). `.reset_index()` turns the two group labels back into normal columns.

In [ ]:
monthly = data.groupby(["YEAR", "MONTH"])["TEMP_C"].mean().reset_index()

print(f"Number of months: {len(monthly)}")
monthly.head()

## Seasons

In Lecture 3.4 we stored the season of each month in a dictionary. The pandas method `.map()` looks up **every** value of a column in such a dictionary:

In [ ]:
# The season of each month (1 = January, ..., 12 = December)
season_of_month = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

monthly["SEASON"] = monthly["MONTH"].map(season_of_month)
monthly.head()

### The December problem

Winter consists of December, January and February. But December 2018 and January-February 2019 are in different calendar years, even though they belong to the **same winter**. If we simply grouped by `YEAR` and `SEASON`, the "winter of 2018" would contain January and February 2018 and December 2018, which belong to two different winters.

The usual convention (also in the project) is to name a winter after the year of its January and February, so **December counts towards the next year's winter**. We create a new column `SEASON_YEAR` that equals `YEAR`, except for December, where it is `YEAR + 1`:

In [ ]:
# Start with the calendar year ...
monthly["SEASON_YEAR"] = monthly["YEAR"]

# ... and add one year for every December
december = monthly["MONTH"] == 12
monthly.loc[december, "SEASON_YEAR"] = monthly.loc[december, "YEAR"] + 1

# Show October 2018 - March 2019
monthly.iloc[9:15]

December 2018 now belongs to the winter of 2019, together with January and February 2019.

## Grouping by two keys: seasonal means per year

Now we group by `SEASON_YEAR` **and** `SEASON`. With `.agg(["mean", "count"])` we calculate two statistics at once: the mean temperature and the number of months in each group.

In [ ]:
seasonal = monthly.groupby(["SEASON_YEAR", "SEASON"])["TEMP_C"].agg(["mean", "count"])
seasonal

Each row now has **two** labels, for example `(2019, Winter)`. This is called a *MultiIndex*.

Look at the `count` column. The winter of 2018 contains only two months (January and February 2018; December 2017 is not in the data), and the winter of 2021 only one (December 2020). Such **incomplete seasons** are not comparable with the others. The winter of 2018 lacks December, usually the mildest of the three winter months, so its mean is too low; a winter without January or February would look too warm. We keep only the seasons that contain all three months:

In [ ]:
complete = seasonal[seasonal["count"] == 3]
complete

> **Note:** each seasonal mean here is the mean of three *monthly* means, so every month counts equally, although February is a little shorter than the other months. This is the natural approach for monthly data such as the project data.

A table with one row per year and one column per season is easier to read and to plot. `.unstack()` moves the inner label (`SEASON`) into the columns. We also put the seasons in their natural order (pandas sorts text alphabetically):

In [ ]:
table = complete["mean"].unstack()
table = table[["Winter", "Spring", "Summer", "Autumn"]]
table.round(1)

The empty cell (`NaN`) is the incomplete winter of 2018, which we left out. The incomplete winter of 2021 has disappeared completely: 2021 has no complete season, so the table has no row for it.

> ⚠️ **Common Pitfall: `.map()` gives `NaN` for values that are not in the dictionary**
>
> If a value has no key in the dictionary, `.map()` returns `NaN` without an error. For example, if the months were stored as text (`"01"`, `"02"`, ...), *every* season would be `NaN`, because the dictionary keys are the numbers 1, 2, .... Check the result with `.isna().sum()`, or look at a few rows.

## Repeating the steps for all stations

To compare the five cities, we put the steps into two functions (Lecture 4.1) and loop over the files with `glob` (Lecture 6.1).

In [ ]:
import glob
import os


def monthly_means(fp):
    """Read one station file and return its monthly mean temperatures (°C).

    The returned DataFrame has the columns YEAR, MONTH, TEMP_C, SEASON and SEASON_YEAR.
    December counts towards the winter of the next year.
    """
    data = pd.read_csv(fp, sep=r"\s+", usecols=["YRMODAHRMN", "Temperature"], na_values=na_values)
    data["TIME"] = pd.to_datetime(data["YRMODAHRMN"].astype(str), format="%Y%m%d%H%M")
    data["YEAR"] = data["TIME"].dt.year
    data["MONTH"] = data["TIME"].dt.month
    data["TEMP_C"] = (data["Temperature"] - 32) / 1.8

    monthly = data.groupby(["YEAR", "MONTH"])["TEMP_C"].mean().reset_index()
    monthly["SEASON"] = monthly["MONTH"].map(season_of_month)
    monthly["SEASON_YEAR"] = monthly["YEAR"]
    december = monthly["MONTH"] == 12
    monthly.loc[december, "SEASON_YEAR"] = monthly.loc[december, "YEAR"] + 1
    return monthly


def seasonal_table(monthly):
    """Return the mean temperature of each complete season: one row per season-year, one column per season."""
    seasonal = monthly.groupby(["SEASON_YEAR", "SEASON"])["TEMP_C"].agg(["mean", "count"])
    complete = seasonal[seasonal["count"] == 3]
    return complete["mean"].unstack()[["Winter", "Spring", "Summer", "Autumn"]]

In [ ]:
# All station files, in alphabetical order
file_list = sorted(glob.glob("dataset/*txt"))

# Monthly means of every city, stored in a dictionary: city name -> DataFrame
all_monthly = {}
for fp in file_list:
    # The city name is the part of the file name before the first "_"
    city = os.path.basename(fp).split("_")[0]
    all_monthly[city] = monthly_means(fp)

print(list(all_monthly.keys()))

`os.path.basename()` removes the folder from a file path (`dataset/Malmo_continuous_final.txt` becomes `Malmo_continuous_final.txt`). It works with the `/` used on macOS and Linux as well as the `\` used on Windows.

## One panel per season

In Lecture 7.2 we drew four panels with four almost identical lines of code. With a loop we write the plotting code only once. `plt.subplots()` returns the panels (`axes`) as a 2 × 2 grid; `.flatten()` turns the grid into a simple sequence of four panels, and `zip()` pairs each panel with a season:

In [ ]:
seasons = ["Winter", "Spring", "Summer", "Autumn"]

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))

for ax, season in zip(axes.flatten(), seasons):
    # One line per city in this panel
    for city, monthly in all_monthly.items():
        table = seasonal_table(monthly)
        ax.plot(table.index, table[season], marker="o", label=city)

    ax.set_title(season)
    ax.set_ylabel("Mean temperature (°C)")
    ax.set_xticks([2018, 2019, 2020])
    ax.grid(True)

# One legend is enough, and a title for the whole figure
axes[0][0].legend()
fig.suptitle("Seasonal mean temperature in five cities (simulated data)")
plt.show()

The winter panel has no point for 2018, because that winter is incomplete. In all five cities, the simulated data show the warmest summer in 2018 and the mildest winter in 2020. Malmö and Gothenburg, in the south-west, have milder winters than Stockholm, Uppsala and Västerås.

> **Tip:** in the project you have one city and about 270 years. The same loop works: plot one line (the city) in each panel, and leave out `marker="o"`, which would hide the line when there are many points.

## Summary

This lesson brought together the techniques that the project needs:

- `pd.to_datetime(..., format=...)` turns date text into real dates, and `.dt.year` / `.dt.month` give their parts.
- A **dictionary** and `.map()` assign a season to every month.
- A **season-year** column makes December count towards the next winter.
- `groupby()` on **two keys** with `.agg(["mean", "count"])` gives seasonal means per year, and the count reveals **incomplete seasons**.
- `.unstack()` turns the result into a table with one column per season.
- A `for` loop with `zip(axes.flatten(), seasons)` fills a grid of panels.

| Watch out for | Reminder |
| --- | --- |
| `%m` vs `%M` | Month vs minute in `format`. Check a few converted dates. |
| December | Belongs to the **next** year's winter: use a season-year column. |
| Incomplete seasons | Count the months in each group and leave out seasons with fewer than 3. |
| `.map()` | Values that are not keys in the dictionary become `NaN`, without an error. |